In [ ]:
import math
import os
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from IPython.display import Latex, display

from structural_tools.seismic.parameters import (
    RiskCategory,
    SeismicParameters,
    SiteClass,
)

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ","})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 3)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")
# handcalcs.set_option("math_environment_start", "aligned")
# handcalcs.set_option("math_environment_end", "aligned")


def set_params_columns(num_cols: int):
    handcalcs.set_option("param_columns", num_cols)


def check_value(value: float, check_value: float = 1, inequality: str = "leq"):
    pass_latex = r"\textbf{\color{OK}OK}\ {\color{OK}\checkmark}"
    fail_latex = r"\textbf{{\color{NG}NG !}}"
    outcome = fail_latex
    operator = "ERROR"
    match inequality:
        case "leq" | "<=":
            if value <= check_value:
                outcome = pass_latex
                operator = "\\leq"
            else:
                operator = "\\gt"
        case "lt" | "<":
            if value < check_value:
                outcome = pass_latex
                operator = "\\lt"
            else:
                operator = "\\geq"
        case "geq" | ">=":
            if value >= check_value:
                outcome = pass_latex
                operator = "\\geq"
            else:
                operator = "\\lt"
        case "gt" | ">":
            if value > check_value:
                outcome = pass_latex
                operator = "\\gt"
            else:
                operator = "\\leq"
        case "eq" | "=":
            if value == check_value:
                outcome = pass_latex
                operator = "="
            else:
                operator = "\\neq"
    return f"{value:.3g} {operator} {check_value:.3g} \\quad {outcome}"


def display_table(
    styler: pd.io.formats.style.Styler | None = None,
    hrules=True,
    position="H",
    **kwargs,
) -> Latex | None:
    """Displays a table if run in a Jupyter ipynb, or returns LaTeX code if run by nbconvert when exporting to PDF

    Args:
        styler (pd.io.formats.style.Styler): Formatted table styler created by the `create_formatted_table` function
        position_float (str, optional): See pandas documentation for Styler.to_latex(). Defaults to "centering".
        hrules (bool, optional): See pandas documentation for Styler.to_latex(). Defaults to True.
        position (str, optional): See pandas documentation for Styler.to_latex(). Defaults to "H".

    Returns:
        _type_: _description_
    """
    # Need to use the "export_to_*" scripts to set the NBCONVERT environment variable for this to work,
    # otherwise it will just display the dataframe as normal without LaTeX formatting
    if "NBCONVERT" in os.environ:
        return Latex(
            styler.to_latex(
                hrules=hrules,
                position=position,
                **kwargs,
            )
        )
    else:
        display(styler.data)


def sig_figs(x: float, sig_figs: int):
    """
    Rounds a number to number of significant figures
    Parameters:
    - x - the number to be rounded
    - precision (integer) - the number of significant figures
    Returns:
    - float
    """

    if pd.isna(x):
        return ""

    if x == 0:
        return float(0)

    x = float(x)
    sig_figs = int(sig_figs)

    if 1e-3 <= abs(x) < 1e6:
        decimals = sig_figs - int(math.floor(math.log10(abs(x)))) - 1
        decimals = max(decimals, 0)
        return f"{x:.{decimals}f}"

    return round(x, -int(math.floor(math.log10(abs(x)))) + (sig_figs - 1))

# Effective Seismic Weight Calculation

## Unit Weights

In [70]:
set_params_columns(4)

In [71]:
%%render params
U_corridor = 35 / 1000 * ksf
U_floor = 30 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

In [72]:
%%render param
L_floor = 230 * ft
w_floor = (30 + 9 / 12) * ft
L_corridor = 230 * ft
w_corridor = 5.5 * ft
w_building = (w_floor * 2) + w_corridor
h_wall__1 = 11.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10 * ft
L_jog = 5 * ft
P_wall = (2 * L_floor) + (2 * w_building) + (2 * L_jog)

<IPython.core.display.Latex object>

In [73]:
%%render 
A_floor = 2 * (L_floor * w_floor)
# This is a slight overestimate as the corridor doesn't stretch the full length of the building
A_corridor = L_corridor * w_corridor
A_level = L_floor * w_building
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall

<IPython.core.display.Latex object>

## Component Weights

In [74]:
%%render long
W_floor__2 = U_floor * A_floor
W_floor__3 = W_floor__2
W_floor__4 = W_floor__2
W_corridor__2 = U_corridor * A_corridor
W_corridor__3 = W_corridor__2
W_corridor__4 = W_corridor__2
W_roof = U_roof * A_level
# W_truss = U_truss * A_level
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__roof = U_wall * (A_wall__4 / 2)

<IPython.core.display.Latex object>

## Level Weights

In [75]:
%%render long
W_level__2 = W_floor__2 + W_corridor__2 + W_wall__2
W_level__3 = W_floor__3 + W_corridor__3 + W_wall__3
W_level__4 = W_floor__4 + W_corridor__4 + W_wall__4
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof  # + Wtruss
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [76]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(
    s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D
)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [77]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [78]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [79]:
set_params_columns(1)

In [80]:
%%render params
h_top__roof = (57 + 8 / 12) * ft
h_bottom__roof = 45.5 * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [81]:
%%render
h_n__check = check_value(h_n, 65 * ft, "leq")

<IPython.core.display.Latex object>

We will approximate the building period.

In [82]:
h_n = float(h_n)

In [83]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [84]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [85]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [86]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")    # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [87]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [88]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [89]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [90]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [91]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [92]:
# Load in basic data
df = pd.DataFrame(
    {
        "Level": ["Ground", "Level 2", "Level 3", "Level 4", "Roof"],
        "floor_height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            0,
        ],
        "floor_weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__roof),
        ],
    },
)
df = df.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in df.floor_height[:-1]:
    elevations.append(height + elevations[-1])
df.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
df["wxhxk"] = df.floor_weight * df.elevation ** float(k)
cumulative_wxhxk = df.wxhxk.sum()
df["C_vx"] = df.wxhxk / cumulative_wxhxk
df["F_x"] = df.C_vx * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(df.F_x[1:])):
    story_shears.insert(0, shear + story_shears[0])
df["V_x"] = story_shears

# Calc overturning moments
floor_moments = df.V_x * df.floor_height
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
df["OTM"] = overturning_moments

In [93]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor_height": "$h_{floor}$ [ft]",
    "floor_weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}
df_display = df.copy()
df_display.rename(columns=column_name_map, errors="raise", inplace=True)

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [94]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
styler = df_display.iloc[::-1].style
styler = styler.format(lambda value: sig_figs(value, sig_figs=3))
display_table(
    styler=styler,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,421.490,42.5,17913.3250,0.332663,71.308726,0.000000,0.000000
Level 4,10.0,542.917,32.5,17644.8025,0.327677,70.239801,71.308726,713.087262
Level 3,10.5,544.729,22.0,11984.0380,0.222552,47.705632,141.548528,2199.346802
Level 2,10.5,548.353,11.5,6306.0595,0.117108,25.102937,189.254159,4186.515474
Ground,11.5,0.000,0.0,0.0000,0.000000,0.000000,214.357096,6651.622082


## Diaphragm Analysis

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

### Fully Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (corridors are 17% heavier than the rest of the floors, for instance), but the vast majority of the area of each level is typical floor weight (30 psf) instead of corridor weight (35 psf).

$$
f_{p,x} = F_x / A_{level}
$$

In [95]:
%%render
A_level
f_p__roof = df.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__4 = df.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = df.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = df.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

The walls that are close to inline (A-5 and A-5.1, A-6 and A-6.1, and A-9 and A-9.1) in reality will be treated as inline and have collectors to transfer the shear load from the diaphragm to those walls.

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/image.png){width=60% #fig:diaphragm_trib_area}

In [96]:
# Plan dimensions
l_A1__A2 = (31 + 11 / 12) * ft
l_A2__A3 = (17 + 11 / 12) * ft
l_A3__A4 = (26 + 4 / 12) * ft
l_A4__A5 = (21 + 4 / 12) * ft
l_A5__A6 = (24 + 8 / 12) * ft
l_A6__A7 = (34 + 4 / 12) * ft
l_A7__A8 = (26 + 1 / 12) * ft
l_A8__A9 = (21 + 4 / 12) * ft
l_A9__A10 = (31 + 1 / 12) * ft

l_A1__A24 = (40 + 1 / 12) * ft
l_A24__A35 = (23 + 8 / 12) * ft
l_A35__A51 = (34 + 4 / 12) * ft
l_A51__A61 = (24 + 8 / 12) * ft
l_A61__A68 = (24 + 8 / 12) * ft
l_A68__A82 = (38 + 3 / 12) * ft
l_A82__A89 = (17 + 11 / 12) * ft
l_A89__A10 = (31 + 5 / 12) * ft

l_A37__A4 = (8 + 10 / 12) * ft
l_A3__A37 = l_A3__A4 - l_A37__A4
l_A78__A82 = (8 + 10 / 12) * ft
l_A68__A78 = l_A68__A82 - l_A78__A82

d_trib = 67 / 2 * ft

In [ ]:
# Wall tributary widths
t_w__A2 = l_A1__A2 + (l_A2__A3 + l_A3__A37) / 2
t_w__A24 = (l_A1__A24 - L_jog) + ((l_A24__A35 + l_A35__A51) / 2)
t_w__A37 = ((l_A2__A3 + l_A3__A37) / 2) + (l_A37__A4 / 2)
t_w__A4 = (l_A37__A4 / 2) + (l_A4__A5 / 2)
t_w__A5 = (l_A4__A5 / 2) + (l_A5__A6 / 2)
t_w__A51 = ((l_A24__A35 + l_A35__A51) / 2) + (l_A51__A61 / 2)
t_w__A6 = (l_A5__A6 / 2) + (l_A6__A7 / 2)
t_w__A61 = (l_A51__A61 / 2) + (l_A61__A68 / 2)
t_w__A68 = (l_A61__A68 / 2) + (l_A68__A78 / 2)
t_w__A7 = (l_A6__A7 / 2) + (l_A7__A8 / 2)
t_w__A78 = (l_A68__A78 / 2) + (l_A78__A82 / 2)
t_w__A8 = (l_A7__A8 / 2) + (l_A8__A9 / 2)
t_w__A82 = (l_A78__A82 / 2) + (l_A82__A89 / 2)
t_w__A89 = (l_A82__A89 / 2) + l_A89__A10
t_w__A9 = (l_A8__A9 / 2) + (l_A9__A10 - L_jog)


<IPython.core.display.Latex object>

In [ ]:
levels = ["Roof", "Level 4", "Level 3", "Level 2"]
walls = [
    "A-2",
    "A-2.4",
    "A-3.7",
    "A-4",
    "A-5",
    "A-5.1",
    "A-6",
    "A-6.1",
    "A-6.8",
    "A-7",
    "A-7.8",
    "A-8",
    "A-8.2",
    "A-8.9",
    "A-9",
]

index = pd.MultiIndex.from_product([levels, walls], names=["Level", "Wall"])

df = pd.DataFrame(index=index)
df.loc["Roof", "tributary weight"] = float(f_p__roof)
df.loc["Level 4", "tributary weight"] = float(f_p__4)
df.loc["Level 3", "tributary weight"] = float(f_p__3)
df.loc["Level 2", "tributary weight"] = float(f_p__2)

# Add wall tributary lengths to df
wall_trib_widths = {
    "A-2": float(t_w__A2),
    "A-2.4": float(t_w__A24),
    "A-3.7": float(t_w__A37),
    "A-4": float(t_w__A4),
    "A-5": float(t_w__A5),
    "A-5.1": float(t_w__A51),
    "A-6": float(t_w__A6),
    "A-6.1": float(t_w__A61),
    "A-6.8": float(t_w__A68),
    "A-7": float(t_w__A7),
    "A-7.8": float(t_w__A78),
    "A-8": float(t_w__A8),
    "A-8.2": float(t_w__A82),
    "A-8.9": float(t_w__A89),
    "A-9": float(t_w__A9),
}
df["tributary width"] = df.index.get_level_values("Wall").map(wall_trib_widths)
df["tributary depth"] = float(d_trib)
df["tributary area"] = df["tributary depth"] * df["tributary width"]
level_area = df["tributary area"].sum() / 4
assert np.isclose(level_area, float(A_level))
df["floor shear"] = df["tributary area"] * df["tributary weight"]
df["cumulative shear"] = df.groupby(level="Wall")["floor shear"].cumsum()

l_aa__ab = 30 + 9 / 12
l_ac__ad = 30 + 9 / 12
l_stairwall = 20
wall_lengths = {
    "A-2": l_aa__ab,
    "A-2.4": l_ac__ad,
    "A-3.7": l_stairwall,
    "A-4": l_stairwall,
    "A-5": l_aa__ab,
    "A-5.1": l_ac__ad,
    "A-6": l_aa__ab,
    "A-6.1": l_ac__ad,
    "A-6.8": l_ac__ad,
    "A-7": l_aa__ab,
    "A-7.8": l_stairwall,
    "A-8": l_aa__ab,
    "A-8.2": l_stairwall,
    "A-8.9": l_ac__ad,
    "A-9": l_aa__ab,
}
df["wall length"] = df.index.get_level_values("Wall").map(wall_lengths)
df["unit shear"] = df["cumulative shear"] / df["wall length"]
load_factor = 0.7
df["factored unit shear"] = df["unit shear"] * load_factor

In [99]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{\\text{trib}}$ [psf]",
    "tributary width": "$l_{\\text{trib}}$ [ft]",
    "tributary depth": "$d_{\\text{trib}}$ [ft]",
    "tributary area": "$A$ [ft$^2$]",
    "floor shear": "$F_x$ [lbf]",
    "cumulative shear": "$F_x$ [lbf]",
    "wall length": "$l_{\\text{wall}}$ [ft]",
    "unit shear": "$v$ [plf]",
    "factored unit shear": "$0.7 v$ [lbf]",
}
df_display = df.copy()
df_display.rename(columns=column_name_map, errors="raise", inplace=True)

In [100]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
styler = df_display.loc["Roof"].style.format(lambda value: sig_figs(value, sig_figs=3))
display_table(
    styler=styler,
    position_float="centering",
    caption="Diaphragm Forces - Roof",
    label=f"tab:diaphragm_forces_roof_ns",
)

,$w_{\text{trib}}$ [psf],$l_{\text{trib}}$ [ft],$d_{\text{trib}}$ [ft],$A$ [ft$^2$],$F_x$ [lbf],$F_x$ [lbf],$l_{\text{wall}}$ [ft],$v$ [plf],$0.7 v$ [lbf]
Wall,,,,,,,,,
A-2,4.627432,49.625000,33.5,1662.437500,7692.816386,7692.816386,30.75,250.172891,175.121023
A-2.4,4.627432,64.083333,33.5,2146.791667,9934.132327,9934.132327,30.75,323.061214,226.142850
A-3.7,4.627432,22.125000,33.5,741.187500,3429.794711,3429.794711,20.00,171.489736,120.042815
A-4,4.627432,15.083333,33.5,505.291667,2338.202797,2338.202797,20.00,116.910140,81.837098
A-5,4.627432,23.000000,33.5,770.500000,3565.436310,3565.436310,30.75,115.949148,81.164404
A-5.1,4.627432,41.333333,33.5,1384.666667,6407.450760,6407.450760,30.75,208.372382,145.860668
A-6,4.627432,29.500000,33.5,988.250000,4573.059615,4573.059615,30.75,148.717386,104.102170
A-6.1,4.627432,24.666667,33.5,826.333333,3823.801260,3823.801260,30.75,124.351260,87.045882
A-6.8,4.627432,27.041667,33.5,905.895833,4191.971314,4191.971314,30.75,136.324270,95.426989


In [101]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
styler = df_display.loc["Level 4"].style.format(
    lambda value: sig_figs(value, sig_figs=3)
)
display_table(
    styler=styler,
    position_float="centering",
    caption="Diaphragm Forces - Level 4",
    label=f"tab:diaphragm_forces_level4_ns",
)

,$w_{\text{trib}}$ [psf],$l_{\text{trib}}$ [ft],$d_{\text{trib}}$ [ft],$A$ [ft$^2$],$F_x$ [lbf],$F_x$ [lbf],$l_{\text{wall}}$ [ft],$v$ [plf],$0.7 v$ [lbf]
Wall,,,,,,,,,
A-2,4.558066,49.625000,33.5,1662.437500,7577.500313,15270.316699,30.75,496.595665,347.616966
A-2.4,4.558066,64.083333,33.5,2146.791667,9785.218708,19719.351036,30.75,641.279708,448.895796
A-3.7,4.558066,22.125000,33.5,741.187500,3378.381752,6808.176463,20.00,340.408823,238.286176
A-4,4.558066,15.083333,33.5,505.291667,2303.152908,4641.355705,20.00,232.067785,162.447450
A-5,4.558066,23.000000,33.5,770.500000,3511.990070,7077.426379,30.75,230.160207,161.112145
A-5.1,4.558066,41.333333,33.5,1384.666667,6311.402444,12718.853204,30.75,413.621242,289.534870
A-6,4.558066,29.500000,33.5,988.250000,4504.509002,9077.568617,30.75,295.205483,206.643838
A-6.1,4.558066,24.666667,33.5,826.333333,3766.482104,7590.283364,30.75,246.838483,172.786938
A-6.8,4.558066,27.041667,33.5,905.895833,4129.133252,8321.104566,30.75,270.605027,189.423519


In [102]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
styler = df_display.loc["Level 3"].style.format(
    lambda value: sig_figs(value, sig_figs=3)
)
display_table(
    styler=styler,
    position_float="centering",
    caption="Diaphragm Forces - Level 3",
    label=f"tab:diaphragm_forces_level3_ns",
)

,$w_{\text{trib}}$ [psf],$l_{\text{trib}}$ [ft],$d_{\text{trib}}$ [ft],$A$ [ft$^2$],$F_x$ [lbf],$F_x$ [lbf],$l_{\text{wall}}$ [ft],$v$ [plf],$0.7 v$ [lbf]
Wall,,,,,,,,,
A-2,3.095758,49.625000,33.5,1662.437500,5146.504286,20416.820986,30.75,663.961658,464.773161
A-2.4,3.095758,64.083333,33.5,2146.791667,6645.947601,26365.298636,30.75,857.408086,600.185660
A-3.7,3.095758,22.125000,33.5,741.187500,2294.537176,9102.713638,20.00,455.135682,318.594977
A-4,3.095758,15.083333,33.5,505.291667,1564.260749,6205.616454,20.00,310.280823,217.196576
A-5,3.095758,23.000000,33.5,770.500000,2385.281584,9462.707963,30.75,307.730340,215.411238
A-5.1,3.095758,41.333333,33.5,1384.666667,4286.592991,17005.446195,30.75,553.022640,387.115848
A-6,3.095758,29.500000,33.5,988.250000,3059.382901,12136.951518,30.75,394.697610,276.288327
A-6.1,3.095758,24.666667,33.5,826.333333,2558.128075,10148.411439,30.75,330.029640,231.020748
A-6.8,3.095758,27.041667,33.5,905.895833,2804.434326,11125.538891,30.75,361.806143,253.264300


In [103]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
styler = df_display.loc["Level 2"].style.format(
    lambda value: sig_figs(value, sig_figs=3)
)
display_table(
    styler=styler,
    position_float="centering",
    caption="Diaphragm Forces - Level 2",
    label=f"tab:diaphragm_forces_level2_ns",
)

,$w_{\text{trib}}$ [psf],$l_{\text{trib}}$ [ft],$d_{\text{trib}}$ [ft],$A$ [ft$^2$],$F_x$ [lbf],$F_x$ [lbf],$l_{\text{wall}}$ [ft],$v$ [plf],$0.7 v$ [lbf]
Wall,,,,,,,,,
A-2,1.629003,49.625000,33.5,1662.437500,2708.115766,23124.936752,30.75,752.030463,526.421324
A-2.4,1.629003,64.083333,33.5,2146.791667,3497.130183,29862.428819,30.75,971.135897,679.795128
A-3.7,1.629003,22.125000,33.5,741.187500,1207.396702,10310.110340,20.00,515.505517,360.853862
A-4,1.629003,15.083333,33.5,505.291667,823.121669,7028.738123,20.00,351.436906,246.005834
A-5,1.629003,23.000000,33.5,770.500000,1255.146854,10717.854817,30.75,348.548124,243.983687
A-5.1,1.629003,41.333333,33.5,1384.666667,2255.626230,19261.072424,30.75,626.376339,438.463437
A-6,1.629003,29.500000,33.5,988.250000,1609.862269,13746.813787,30.75,447.050855,312.935598
A-6.1,1.629003,24.666667,33.5,826.333333,1346.099524,11494.510963,30.75,373.805235,261.663664
A-6.8,1.629003,27.041667,33.5,905.895833,1475.707080,12601.245971,30.75,409.796617,286.857632


## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4. \

The north-south direction has many bays of shear walls, while the east-west direction is only a single bay of shear walls with cantilever diaphragms. Redundancy factors are shown in \autoref{tab:redundancy_factors}

Table: Redundancy Factors {#tab:redundancy_factors}

| Loading Direction | Redundancy Factor, $\rho$ |
| :---: | :---: |
| North-South | 1.0 |
| East-West | 1.3 |